In [ ]:
import sys
from pathlib import Path
from IPython.display import HTML, display

SRC          = str((Path('..') / 'src').resolve())
CIRCUIT_HTML = str((Path('..') / 'manuscript' / 'hippocampal_circuit_neutral.html').resolve())
sys.path.insert(0, SRC)

with open(CIRCUIT_HTML) as f:
    display(HTML(f.read()))

In [ ]:
import torch
from layer import L_DG

## Step 3: L_DG — Dentate Gyrus

**Role**: Pattern separation. Maps ECin's distributed input to a highly sparse (~1% active)
orthogonal representation, so that similar ECin patterns produce distinct DG codes.

**Key parameters (Schapiro 2017 §2.a.iii)**:
- `ecin_frac = 0.25`: each DG unit receives from 25% of ECin units
- `k_frac = 0.01`: ~1% of DG units active after kWTA
- No recurrent connections (unlike CA3)

**Understanding check**: Why 1% sparsity?  
→ Forces near-orthogonal DG codes per episode. CA3 then stores distinct attractors per episode.
If sparsity were 50% (like CA3), DG patterns would heavily overlap → CA3 attractors interfere → pattern completion fails.

In [ ]:
# Instantiate and inspect structure
# n_input=15 (Schapiro 2017 n_items); n_DG=100
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25, use_euler=True)

print(f"W shape      : {dg.W.shape}")            # (15, 100)
print(f"mask shape   : {dg.mask.shape}")         # (15, 100)
print(f"mask density : {dg.mask.mean():.3f}")    # ~0.25
print(f"n_active target: {max(1, int(dg.k_frac * dg.n_DG))}")  # 1 unit at k_frac=0.01

In [ ]:
# Forward pass: single ECin pattern → sparse DG output
torch.manual_seed(42)
dg.W.data = torch.randn(15, 100) * 0.1

a_ecin = torch.zeros(15)
a_ecin[0] = 1.0

dg.reset()
act = dg(a_ecin)

n_active = (act > 0).sum().item()
print(f"Active DG units: {n_active} / {dg.n_DG} = {n_active / dg.n_DG:.3f}")
print(f"Max activity   : {act.max().item():.4f}")
print(f"Active indices : {(act > 0).nonzero(as_tuple=True)[0].tolist()}")

In [ ]:
# Pattern separation: two similar ECin patterns → very different DG codes
# Item A: units 0 and 1 active (moving window: curr=1.0, prev=0.9)
a_A = torch.zeros(15); a_A[0] = 1.0; a_A[1] = 0.9
# Item B: units 1 and 2 active (shares unit 1 with A)
a_B = torch.zeros(15); a_B[1] = 1.0; a_B[2] = 0.9

cos_ecin = torch.nn.functional.cosine_similarity(a_A.unsqueeze(0), a_B.unsqueeze(0)).item()

dg.reset(); act_A = dg(a_A).clone()
dg.reset(); act_B = dg(a_B).clone()

cos_dg = torch.nn.functional.cosine_similarity(act_A.unsqueeze(0), act_B.unsqueeze(0)).item()

print(f"ECin cosine similarity (A, B): {cos_ecin:.3f}")
print(f"DG   cosine similarity (A, B): {cos_dg:.3f}")
print("Pattern separation: DG similarity should be much lower than ECin similarity.")

In [ ]:
# CHL weight update — masked
dg.reset(); act_minus = dg(a_A).clone()
dg.reset(); act_plus  = dg(a_B).clone()

W_before = dg.W.data.clone()
dg.update_weights(a_ECin_minus=a_A, a_ECin_plus=a_B,
                  a_DG_minus=act_minus, a_DG_plus=act_plus, lr=0.4)
delta_W = dg.W.data - W_before

print(f"Weights changed at masked positions  : {(delta_W[dg.mask.bool()] != 0).sum().item()}")
print(f"Weights changed at unmasked positions: {(delta_W[~dg.mask.bool()] != 0).sum().item()}")
print("Unmasked positions must remain zero.")